In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import os
import sys
import astropy as ast
from astropy.io import ascii
from astropy.table import Table
import matplotlib.pyplot as plt
import pandas as pd
import glob
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.colors import LogNorm, Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
import matplotlib.path as mpath
from pathlib import Path
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import MultipleLocator
from matplotlib.ticker import AutoMinorLocator
fourpointstar = mpath.Path.unit_regular_star(4)
sevenpointstar = mpath.Path.unit_regular_star(7)
eightpointstar = mpath.Path.unit_regular_star(8)
from metrics import fit_plotter_flux, compute_lomb_scargle, plot_periodogram, info, plot_phase_fold_binned, plot_phase_fold

In [ ]:
stats = pd.read_csv('../summary_results03162026.csv')
df_smc = stats[stats['RA'] < 40]
df_lmc = stats[stats['RA'] > 40]
coords = pd.read_csv('../merged_smc_lmc_coords_all.csv', comment='#', sep="\\s+", names=['RA', 'DEC'])
hrd_zoning = pd.read_csv('./HRD_zoning_priority_03222026.csv')
# temps = pd.read_csv('../synth_phot_temp_estimation/ysg_temp_fitting_summary_v10_prefinal.csv')
temps = pd.read_csv('../synth_phot_temp_estimation/ysg_temp_fitting_summary_03152026.csv')
var = pd.read_csv('../summary_results03162026.csv')
hrd_priority = pd.read_csv("HRD_zoning_priority_03202026.csv")
magellan_spectroscopy_orig = pd.read_csv("../magellan_spectroscopy/magellan_spectroscopy_to_original_matches.csv")
magellan_spectroscopy_prefinal = pd.read_csv("../magellan_spectroscopy/magellan_spectroscopy_to_prefinal_matches.csv")
plt.rcParams['font.family'] = 'serif'

In [ ]:
print(((var['logT'] > 3.62) & (var['logL'] > 4.0) & (var['logT'] < 4.0) & (var['best_period'].notnull())).sum())
print(((temps['final_logT_mean'] > 3.62) & (temps['final_logL_mean'] > 4.0) & (temps['final_logT_mean'] < 4.0)).sum())

In [ ]:
# Read VizieR VOTable files with embedded CSV payloads.
from io import StringIO
import re

def read_vizier_votable_csv(path):
    with open(path, 'r', encoding='utf-8') as f:
        text = f.read()
    match = re.search(r'<!\[CDATA\[(.*?)\]\]>', text, flags=re.S)
    if match is None:
        raise ValueError(f'No CDATA CSV block found in {path}')
    csv_block = match.group(1).strip()
    df = pd.read_csv(StringIO(csv_block), sep=';', skiprows=[1, 2], engine='python')
    if len(df) == 0:
        raise ValueError(f'Parsed zero rows from {path}')
    return df

rsg_smc = read_vizier_votable_csv('RSG_SMC_massey23.tsv')
rsg_lmc = read_vizier_votable_csv('RSG_LMC_massey23.tsv')

# Ensure numeric columns for plotting.
for _df in (rsg_smc, rsg_lmc):
    _df['Temp'] = pd.to_numeric(_df['Temp'], errors='coerce')
    _df['LogLum'] = pd.to_numeric(_df['LogLum'], errors='coerce')

def truncate_colormap(cmap, minval=0.0, maxval=1.0, n=100):
    new_cmap = mpl.colors.LinearSegmentedColormap.from_list(
        f'trunc({cmap.name},{minval:.2f},{maxval:.2f})',
        cmap(np.linspace(minval, maxval, n))
    )
    return new_cmap

In [ ]:
offsets = pd.read_csv("../offsets.csv")
print(offsets['discard_lc'].sum(), "light curves discarded due to bad offsets")
print(var['alarm_level_flag'].sum(), "light curves discarded due to no significant period")
print(var['logT'].isna().sum(), "light curves discarded due to no temperature estimate")
print((var['alarm_level_flag']==0).sum(), "light curves kept")
print(((var['SIMBAD_maintype']== 'ClassicalCep') & (var['alarm_level_flag']==0)).sum())
print(((var['SIMBAD_maintype']== 'LongPeriodV*') & (var['alarm_level_flag']==0)).sum())
print(((var['SIMBAD_maintype']== 'ChemPec*') & (var['alarm_level_flag']==0)).sum())
print(((var['SIMBAD_maintype']== 'EmLine*') & (var['alarm_level_flag']==0)).sum())
print(((var['SIMBAD_maintype']== 'PulsV*') & (var['alarm_level_flag']==0)).sum())
print(((var['SIMBAD_maintype']== 'EclBin') & (var['alarm_level_flag']==0)).sum())
print(((var['SIMBAD_maintype']== 'HighPM*') & (var['alarm_level_flag']==0)).sum())
print(((var['SIMBAD_maintype']== 'WolfRayet*') & (var['alarm_level_flag']==0)).sum())

In [ ]:
def basic_HRD(column, cbar_label='Largest Amplitude [mJy]'):
    plt.rcParams['font.family'] = 'serif'
    fig, ax = plt.subplots(figsize=(8, 6))
    c = stats[column]
    mask = (stats['alarm_level_flag'] == 0).values
    nonvar_mask = (stats['alarm_level_flag'] == 1).values
    vmax = np.max(c[mask])
    vmax = 0.2
    # vmax = 80
    vmin = 0
    # norm_smc = Normalize(vmin=np.min(smc_c[mask_smc]), vmax=np.max(smc_c[mask_smc]))

    sc01 = ax.scatter(stats['logT'].values[nonvar_mask], stats['logL'].values[nonvar_mask], c='gray', alpha=0.7, edgecolors='none', label='Non-periodic')
    # LMC
    sc1 = ax.scatter(stats['logT'].values[mask], stats['logL'].values[mask],
                    c=c[mask], cmap='inferno_r', norm=Normalize(vmin=np.min(c[mask]), vmax=vmax),
                    alpha=0.7, zorder=10, edgecolors='black', label='Periodic')
    # ax.set_xlim(3.9, 3.95)
    # ax.set_ylim(4.2,4.4)
    ax.invert_xaxis()
    ax.set_xlabel('log(T/[K])', fontsize=18) # /K
    ax.set_ylabel('log(L/L$_{\\odot}$)', fontsize=18) #L_{\odot} 
    cbar = fig.colorbar(sc1, ax=ax, orientation='vertical')
    cbar.set_label(cbar_label, fontsize=18)
    cbar.ax.tick_params(labelsize=16)

    ticks = cbar.get_ticks()
    # tick_labels = [f'{tick:.0f}' for tick in ticks]
    # tick_labels[-1] = '80+'

    tick_labels = [f'{tick:.2f}' for tick in ticks]
    tick_labels[-1] = '0.2+'

    # cbar.set_ticks(ticks)
    cbar.set_ticklabels(tick_labels)
    plt.legend(loc='lower left', fontsize=14)
    plt.tick_params(axis='both', which='both', direction = 'in', labelsize=16)
    plt.grid(True, alpha=0.2)
    # plt.savefig(f'hr_diagrams/HR_{column}_shortened.png', dpi=300, bbox_inches='tight')
    plt.show()

basic_HRD('mag_amplitude_1', cbar_label='Largest Amplitude [mags]')# basic_HRD('best_period', cbar_label='Dominant Period [days]')


In [ ]:
# loading in evolutionary models
eep_directory_lmc = '../../MIST_models_LMC/'
evol_files_lmc = glob.glob(f'{eep_directory_lmc}*.track.*')
# order files by mass 
evol_files_lmc = sorted(evol_files_lmc, key=lambda x: float(x.split('/')[-1].split('M')[0]))
print(evol_files_lmc)

# loading in evolutionary models
eep_directory_smc = '../../MIST_models_SMC/'
evol_files_smc = glob.glob(f'{eep_directory_smc}*.track.*')
# order files by mass 
evol_files_smc = sorted(evol_files_smc, key=lambda x: float(x.split('/')[-1].split('M')[0]))
print(evol_files_smc)

def Open_File(file_name):
    # Get the column names
    row = open(file_name, 'r').readlines()
    # It's the last comment in the file so get all the comments
    comments = [x for x in row if '#' in x  ]
    # Split the last row into individual keys, exclude the '#'
    columns = comments[-1].split()[1:]
    # Strip any blank spaces
    columns = [x.strip(' ') for x in columns]
    # Read in as pandas dataframe
    df = pd.read_csv(file_name, comment='#',names = columns,delimiter='\\s+')
    return df

df = Open_File(evol_files_lmc[0])

# Show the first few rows to check that everything looks okay
df.head()

In [ ]:
# i just selected some colors from magma color map
palette=['#013156', '#2d3a6c', '#543f7b', '#7b4283', '#a14582', '#c24a79', '#dc5769', '#ec6d55', '#f08a3e', "#d49517","#013156"]

In [ ]:
def Get_MS(mist_df):
    # Phase = 0 corresponds to the main sequence
    MS = mist_df.loc[mist_df['phase'] == 0.0]
    return MS

def get_RGB(mist_df):
    # Phase = 2 corresponds to the RGB
    RGB = mist_df.loc[mist_df['phase'] == 2.0]
    return RGB

def get_phase(mist_df):
    # Phase = -1 is pre-main sequence
    # Phase = 6 is post-AGB
    # plot everything else 
    RGB = mist_df.loc[(mist_df['phase'] != -1.0) & (mist_df['phase'] != 6.0)]
    return RGB

In [ ]:
def plot_evol_tracks_multiple(files, lmc=True):
    '''
    Create two similar HRD subplots side by side, with separate code blocks
    so each panel can be customized independently later.
    '''
    fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(16, 6), sharex=True, sharey=True)

    cmap = cm.copper
    norm = mcolors.Normalize(vmin=0, vmax=len(files) - 1)

    # -----------------------------
    # Left panel
    # -----------------------------
    for i, file_i in enumerate(files):
        df_i = Open_File(file_i)
        phase_i = get_phase(df_i)
        color = cmap(norm(i))

        ax_left.plot(phase_i['log_Teff'], phase_i['log_L'], "-", color=color, alpha=0.5, zorder=3)

        x_label = 4.071
        idx_near = (phase_i['log_Teff'] - x_label).abs().idxmin()
        y_label = phase_i.loc[idx_near, 'log_L']

        ax_left.annotate(
            str(int(file_i.split('/')[-1].split('.track')[0][1:3])) + r" M$_\odot$",
            xy=(x_label, y_label),
            xytext=(0, 2),
            textcoords='offset points',
            ha='center',
            va='bottom',
            fontsize=12,
            color=color,
            zorder=20,
            alpha=0.8,
            annotation_clip=False
        )

    mask_left = var[
        (var['logT'] > 3.62)
        & (var['logT'] < 4.05)
        & (var['logL'] > 3.95)
        & (var['logL'] < 5.48)
        & (var['alarm_level_flag'] == 0)
    ]
    ax_left.vlines(3.62, 3.95, 5.5, color='gray', alpha=0.5, linestyle='--', zorder=20)
    ax_left.scatter(mask_left['logT'], mask_left['logL'], alpha=0.9, color='xkcd:sun yellow', edgecolors='xkcd:copper', label='YSG', zorder=8)
    ax_left.scatter(np.log10(rsg_smc['Temp']), rsg_smc['LogLum'], alpha=0.3, c='tab:red', label='RSG', zorder=6)
    ax_left.scatter(np.log10(rsg_lmc['Temp']), rsg_lmc['LogLum'], alpha=0.3, c='tab:red', zorder=6)

    sn_zorder = 4
    sn_color = "xkcd:cyan"
    ax_left.errorbar(3.825, 5.0, xerr=0.025, yerr=0.05, fmt='o', color=sn_color, label='SN Progenitor', zorder=sn_zorder)
    ax_left.errorbar(3.72, 5.17, xerr=0.08, yerr=0.04, fmt='o', color=sn_color, zorder=sn_zorder)
    ax_left.errorbar(3.98, 5.14, xerr=[[0.16], [0.21]], yerr=[[0.39], [0.22]], fmt='o', color=sn_color, zorder=sn_zorder)
    ax_left.errorbar(3.63, 4.94, xerr=0.01, yerr=0.06, fmt='o', color=sn_color, zorder=sn_zorder)
    ax_left.errorbar(3.78, 4.92, xerr=0.02, yerr=0.2, fmt='o', color=sn_color, zorder=sn_zorder)
    ax_left.errorbar(3.63, 5.1, xerr=0.05, yerr=0.3, fmt='o', color=sn_color, zorder=sn_zorder)

    ax_left.invert_xaxis()
    ax_left.legend(bbox_to_anchor=(1.0, 1), loc='upper left', title_fontsize=16, fontsize=18, markerscale=1.5)
    ax_left.set_xlim(4.1, 3.59)
    ax_left.set_ylim(3.93, 5.48)
    ax_left.tick_params(axis='both', direction='in', labelsize=16)
    ax_left.set_xlabel('log(T/[K])', fontsize=24)
    ax_left.set_ylabel('log(L/[L$_☉$])', fontsize=24)

    # -----------------------------
    # Right panel (currently same as left; edit here independently)
    # -----------------------------
    for i, file_i in enumerate(files):
        df_i = Open_File(file_i)
        phase_i = get_phase(df_i)
        color = cmap(norm(i))

        ax_right.plot(phase_i['log_Teff'], phase_i['log_L'], "-", color=color, alpha=0.5, zorder=3)

        x_label = 4.071
        idx_near = (phase_i['log_Teff'] - x_label).abs().idxmin()
        y_label = phase_i.loc[idx_near, 'log_L']

        ax_right.annotate(
            str(int(file_i.split('/')[-1].split('.track')[0][1:3])) + r" M$_\odot$",
            xy=(x_label, y_label),
            xytext=(0, 2),
            textcoords='offset points',
            ha='center',
            va='bottom',
            fontsize=12,
            color=color,
            zorder=20,
            alpha=0.8,
            annotation_clip=False
        )

    mask_right = var[
        (var['logT'] > 3.62)
        & (var['logT'] < 4.05)
        & (var['logL'] > 3.95)
        & (var['logL'] < 5.48)
        & (var['alarm_level_flag'] == 0)
    ]
    ax_right.vlines(3.62, 3.95, 5.5, color='gray', alpha=0.5, linestyle='--', zorder=20)
    ax_right.scatter(mask_right['logT'], mask_right['logL'], alpha=0.9, color='xkcd:orangey yellow', edgecolors='xkcd:copper', label='YSG', zorder=8)
    ax_right.scatter(np.log10(rsg_smc['Temp']), rsg_smc['LogLum'], alpha=0.3, c='tab:red', label='RSG', zorder=6)
    ax_right.scatter(np.log10(rsg_lmc['Temp']), rsg_lmc['LogLum'], alpha=0.3, c='tab:red', zorder=6)

    ax_right.errorbar(3.825, 5.0, xerr=0.025, yerr=0.05, fmt='o', color=sn_color, label='SNIIb Progenitor', zorder=sn_zorder)
    ax_right.errorbar(3.72, 5.17, xerr=0.08, yerr=0.04, fmt='o', color=sn_color, zorder=sn_zorder)
    ax_right.errorbar(3.98, 5.14, xerr=[[0.16], [0.21]], yerr=[[0.39], [0.22]], fmt='o', color=sn_color, zorder=sn_zorder)
    ax_right.errorbar(3.63, 4.94, xerr=0.01, yerr=0.06, fmt='o', color=sn_color, zorder=sn_zorder)
    ax_right.errorbar(3.78, 4.92, xerr=0.02, yerr=0.2, fmt='o', color=sn_color, zorder=sn_zorder)
    ax_right.errorbar(3.63, 5.1, xerr=0.05, yerr=0.3, fmt='o', color=sn_color, zorder=sn_zorder)

    ax_right.invert_xaxis()
    ax_right.legend(bbox_to_anchor=(1.0, 1), loc='upper left', title_fontsize=16, fontsize=18, markerscale=1.5)
    ax_right.set_xlim(4.1, 3.59)
    ax_right.set_ylim(3.93, 5.48)
    ax_right.tick_params(axis='both', direction='in', labelsize=16)
    ax_right.set_xlabel('log(T/[K])', fontsize=24)
    ax_right.set_ylabel('log(L/[L$_☉$])', fontsize=24)

    fig.tight_layout()

# plot_evol_tracks_multiple(evol_files_smc[:-3][::-1], lmc=False)

# multiple lightcurves

In [ ]:
from view_and_clean import df_extract
from view_and_clean import offset_corrector
box1 = [307,286,756, 1189] #1168
# box4 = [1130,1182,911,978, 1101]
box2 = [716,541,579,1100]
box3 = [780,670,765, 378]
proposed = [307,1130,613,1085,716,953,250,890,156,40,780,522]

def individual_plotter(indices, tail=1, g=True, seeoutliers=False, report=True, bg=None, border_color=None, border_width=2.0):
    if isinstance(indices, (int, np.integer)):
        indices = [int(indices)]
    else:
        indices = [int(i) for i in indices]

    # Use Matplotlib's default color cycle (one color per star).
    cycle_colors = plt.rcParams['axes.prop_cycle'].by_key().get('color', [])
    if not cycle_colors:
        cycle_colors = ['C0', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9']

    band_label = 'g' if g else 'V'

    # First pass: load data and choose a global reference HJD0 so the x-axis is readable.
    loaded = []
    hjd0 = None
    for index in indices:
        # Filtered data (outliers removed by default).
        df, telescopes = offset_corrector(index, g=g, show=False)
        fulldata = None
        if seeoutliers:
            fulldata = df_extract(index, g=g, seeoutliers=True, report=False)[0]
        loaded.append((index, df, fulldata, telescopes))
        candidate = float(np.nanmin(df['HJD'].values))
        if fulldata is not None:
            candidate = min(candidate, float(np.nanmin(fulldata['HJD'].values)))
        hjd0 = candidate if (hjd0 is None) else min(hjd0, candidate)

    # plt.figure(figsize=(12, 20))
    plt.figure(figsize=(12, 5))
    if bg is not None:
        # plt.gcf().patch.set_facecolor(bg)
        plt.gca().set_facecolor(bg)
    if border_color is not None:
        fig = plt.gcf()
        ax = plt.gca()
        for spine in ax.spines.values():
            spine.set_color(border_color)
            spine.set_linewidth(border_width)
        # fig.patch.set_edgecolor(border_color)
        # fig.patch.set_linewidth(border_width)
        fig.set_frameon(True)
    for j, (index, df, fulldata, telescopes) in enumerate(loaded):
        RA = coords['RA'].iloc[index]
        dec = coords['DEC'].iloc[index]

        if report:
            print(f"\nIndex {index} ({RA} {dec})")
            print("Number of observations per telescope:")
            print(df.groupby('telescope').size())

        color = cycle_colors[j % len(cycle_colors)]
        t = df['HJD'] - hjd0
        if index in proposed:
            color = 'cyan'
        else:
            color = 'grey'
            # color=color
        plt.errorbar(
            t,
            df['mag'],
            yerr=df['mag_err'],
            fmt='o',
            markeredgecolor='none',
            markersize=3,
            capsize=0,
            color=color,
            zorder=10,
            label=f"{band_label}-band (idx {index})",
        )


    plt.xlabel(f'HJD - 2.458e6 [days]', fontsize=24)
    plt.ylabel('m$_g$ [mag]', fontsize=24)
    # plt.title(f"{band_label}-band light curves ({len(indices)} stars)")
    plt.legend()
    # plt.grid(True)
    plt.gca().invert_yaxis()  # Invert y-axis since smaller magnitudes are brighter
    plt.tick_params(axis='both', direction='in', labelsize=16)
    # plt.savefig(f'figs/offsets_examples/{RA}{dec}_g.png')
    # plt.savefig('hr_diagrams/box3.png', bbox_inches='tight', dpi=300)
    # If saving, keep the chosen background (otherwise Matplotlib may default to white).
    # Example: plt.savefig('hr_diagrams/box3.png', bbox_inches='tight', dpi=300, facecolor=plt.gcf().get_facecolor())
    plt.show()

# box_star_colours = ['green', 'xkcd:violet', 'xkcd:burnt orange']
# individual_plotter(box1, g=True, seeoutliers=False, report=True, border_color='green', bg = 'xkcd:very light green', border_width=3.0)
# individual_plotter(box2, g=True, seeoutliers=False, report=True, border_color='xkcd:violet', bg = 'xkcd:very light purple', border_width=3.0)
individual_plotter(box3, g=True, seeoutliers=False, report=True, border_color='xkcd:burnt orange', bg = 'xkcd:light peach', border_width=3.0)


In [ ]:
from view_and_clean import df_extract
from view_and_clean import offset_corrector

box1 = [307, 286, 756, 1189]  # 1168
# box4 = [1130,1182,911,978, 1101]
box2 = [716, 541, 579, 1100]
box3 = [780, 670, 765, 378]
proposed = [307, 1130, 613, 1085, 716, 953, 250, 890, 156, 40, 780, 522]

def individual_plotter(
    indices,
    tail=1,
    g=True,
    seeoutliers=False,
    report=True,
    bg=None,
    bg_alpha=0.2,
    border_color=None,
    border_width=2.0,
    ax=None,
    show=True,
    ylabel=None,
    xlabel=None,
    legend=True,
    title=None,
    invert_y=True,
    tick_labelsize=22,
    label_fontsize=24,
    marker_size=3,
    star_color_proposed="xkcd:marigold",
    star_color_other="xkcd:light grey",
):

    if isinstance(indices, (int, np.integer)):
        indices = [int(indices)]
    else:
        indices = [int(i) for i in indices]

    cycle_colors = plt.rcParams['axes.prop_cycle'].by_key().get('color', [])
    if not cycle_colors:
        cycle_colors = ['C0', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9']

    band_label = 'g' if g else 'V'

    created = False
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 16))
        created = True
    else:
        fig = ax.figure

    if bg is not None:
        ax.set_facecolor(mcolors.to_rgba(bg, alpha=float(bg_alpha)))

    if border_color is not None:
        for spine in ax.spines.values():
            spine.set_color(border_color)
            spine.set_linewidth(border_width)

    loaded = []
    hjd0 = None
    for index in indices:
        df, telescopes = offset_corrector(index, g=g, show=False)
        fulldata = None
        if seeoutliers:
            fulldata = df_extract(index, g=g, seeoutliers=True, report=False)[0]

        candidates = []
        if df is not None and 'HJD' in df and len(df) > 0:
            hjd_vals = df['HJD'].dropna().values
            if hjd_vals.size:
                candidates.append(float(np.nanmin(hjd_vals)))
        if fulldata is not None and 'HJD' in fulldata and len(fulldata) > 0:
            hjd_vals = fulldata['HJD'].dropna().values
            if hjd_vals.size:
                candidates.append(float(np.nanmin(hjd_vals)))
        if not candidates:
            if report:
                print(f"Index {index}: no valid HJD values; skipping")
            continue

        candidate = min(candidates)
        hjd0 = candidate if (hjd0 is None) else min(hjd0, candidate)
        loaded.append((index, df, fulldata, telescopes))

    if not loaded:
        raise ValueError("No plottable light-curve data found for the provided indices")

    for j, (index, df, fulldata, telescopes) in enumerate(loaded):
        RA = coords['RA'].iloc[index]
        dec = coords['DEC'].iloc[index]

        if report:
            print(f"\nIndex {index} ({RA} {dec})")
            print("Number of observations per telescope:")
            if df is not None and len(df) > 0 and 'telescope' in df:
                print(df.groupby('telescope').size())
            else:
                print("(no filtered points)")

        color = cycle_colors[j % len(cycle_colors)]
        color = star_color_proposed if (index in proposed) else star_color_other
        if df is not None and len(df) > 0:
            t = df['HJD'] - hjd0
            #mask for values less than 2800
            t = t[t < 2900]
            df = df.loc[t.index]
            ax.errorbar(
                t,
                df['mag'],
                yerr=df['mag_err'],
                fmt='o',
                markeredgecolor='none',
                markersize=marker_size,
                capsize=0,
                color=color,
                zorder=10,
                label=f"{band_label}-band (idx {index})",
            )

        if fulldata is not None and len(fulldata) > 0:
            t_full = fulldata['HJD'] - hjd0
            t = t[t < 2900]
            df = df.loc[t.index]
            ax.plot(
                t_full,
                fulldata['mag'],
                linestyle='none',
                marker='o',
                markersize=max(1, marker_size - 1),
                markeredgecolor='none',
                color='black',
                alpha=0.25,
                zorder=1,
                label=f"{band_label}-band outliers (idx {index})",
            )

    if xlabel is None:
        xlabel = 'HJD - 2.458e6 [days]'
    if ylabel is None:
        ylabel = f'm$_{{{band_label}}}$ [mags]'
    ax.set_xlabel(xlabel, fontsize=label_fontsize)
    ax.set_ylabel(ylabel, fontsize=label_fontsize)
    if title:
        ax.set_title(title)

    # if legend:
    #     ax.legend()

    if invert_y:
        ax.invert_yaxis()
    ax.tick_params(axis='both', direction='in', labelsize=tick_labelsize)

    if created and show:
        plt.show()

    return ax

def plot_boxes_as_subplots(
    boxes,
    *,
    g=True,
    seeoutliers=False,
    report=False,
    # bg_alpha=0.2,
    styles=None,
    figsize=(12, 18),
    sharex=False,
    sharey=False,
    suptitle=None,

):

    n = len(boxes)
    fig, axes = plt.subplots(nrows=n, ncols=1, figsize=figsize, sharex=sharex, sharey=sharey)
    if n == 1:
        axes = [axes]
    if styles is None:
        styles = [{} for _ in boxes]
    for ax, indices, style in zip(axes, boxes, styles):
        individual_plotter(
            indices,
            g=g,
            seeoutliers=seeoutliers,
            report=report,
            bg=style.get('bg'),
            # bg_alpha=bg_alpha,
            border_color=style.get('border_color'),
            border_width=style.get('border_width', 3.0),
            ax=ax,
            show=False,
            title=style.get('title'),
        )
    # if suptitle:
    #     fig.suptitle(suptitle)
    fig.tight_layout()
    plt.savefig('hr_diagrams/box_comparison_gemini1_1col.pdf', bbox_inches='tight', dpi=300)
    plt.show()
    return fig, axes

plot_boxes_as_subplots(
    [box1, box2, box3],
    g=True,
    seeoutliers=False,
    report=True,
    # bg_alpha=0.25,
    figsize=(12, 10),
    styles=[
        {'border_color': 'green', 'bg': 'xkcd:very light green', 'border_width': 3.0},
        {'border_color': 'xkcd:violet', 'bg': 'xkcd:very light purple', 'border_width': 3.0},
        {'border_color': 'xkcd:burnt orange', 'bg': 'xkcd:light peach', 'border_width': 3.0},
    ],
    suptitle=None,
 )

In [ ]:
def plot_hrd(evol_track_files,
    *,
    open_file_func=None,
    var_df=None,
    rsg_smc_df=None,
    rsg_lmc_df=None,
    boxes=None,
    box_styles=None,
    proposed_indices=None,
    individual_plotter_func=None,
    figsize=(10, 10),
    width_ratios=(1.15, 1.0),
    hrd_xlim=(4.12, 3.40),
    hrd_ylim=(3.93, 5.48),
    ysg_box=(3.62, 4.00, 4.00, 5.48),
    savepath=None,
    dpi=300,
    report=False,
):
    if open_file_func is None:
        open_file_func = Open_File
    if var_df is None:
        var_df = var
    if rsg_smc_df is None:
        rsg_smc_df = rsg_smc
    if rsg_lmc_df is None:
        rsg_lmc_df = rsg_lmc
    if boxes is None:
        boxes = [box3, box1, box2]
    if proposed_indices is None:
        proposed_indices = proposed
    if individual_plotter_func is None:
        individual_plotter_func = individual_plotter

    if box_styles is None:
        box_styles = [
            {"border_color": "xkcd:burnt orange", "bg": "xkcd:light peach", "border_width": 3.0},
            {"border_color": "green", "bg": "xkcd:very light green", "border_width": 3.0},
            {"border_color": "xkcd:violet", "bg": "xkcd:very light purple", "border_width": 3.0}
        ]

    if len(boxes) != 3:
        raise ValueError("Expected exactly 3 boxes (e.g., [box1, box2, box3])")
    # fig = plt.figure(figsize=figsize)
    fig, ax_hrd = plt.subplots(figsize=figsize)
    # Slightly larger gap between the left and right columns.
    gs = GridSpec(1, 2, figure=fig, width_ratios=width_ratios, wspace=0.31)

    # -----------------------------
    # Left panel: HRD
    # -----------------------------
    # ax_hrd = fig.add_subplot(gs[0, 0])

    cmap = cm.inferno
    norm = mcolors.Normalize(vmin=0, vmax=len(evol_track_files) - 1)
    new_cmap = truncate_colormap(cmap, 0.0, 0.8, n=len(evol_track_files))

    for i, file_i in enumerate(evol_track_files):
        df_i = open_file_func(file_i)
        phase_i = get_phase(df_i)
        color = new_cmap(norm(i))

        ax_hrd.plot(
            phase_i["log_Teff"],
            phase_i["log_L"],
            "-",
            color=color,
            alpha=0.7,
            zorder=3,
        )

        x_label = 4.069
        idx_near = (phase_i["log_Teff"] - x_label).abs().idxmin()
        y_label = phase_i.loc[idx_near, "log_L"]

        # Mass annotation
        ax_hrd.annotate(
            str(int(file_i.split("/")[-1].split(".track")[0][1:3])) + r" M$_\odot$",
            xy=(x_label, y_label),
            xytext=(0, 2),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=20,
            color=color,
            zorder=20,
            alpha=0.8,
            annotation_clip=False,
        )


    x0, x1, y0, y1 = ysg_box
    mask = var_df[
        (var_df["logT"] > x0)
        & (var_df["logT"] < x1)
        & (var_df["logL"] > y0)
        & (var_df["logL"] < y1)
        # & (var_df["alarm_level_flag"] == 0)
        # & (var_df["alarm_level_flag"].notna())
    ]
    # ax_hrd.fill_betweenx([y0, y1], x0, x1, color="xkcd:yellow tan", alpha=0.2, zorder=1)
    ax_hrd.add_patch(Rectangle((3.62, 4.0),0.38,1.6,facecolor='none',edgecolor='black',
                alpha=0.5,
                zorder=20,
                linewidth=1.2,
                linestyle='dotted',
            ))

    ax_hrd.scatter(
        mask["logT"],
        mask["logL"],
        alpha=0.5,
        # color="xkcd:butter yellow",
        color='grey',
        # edgecolors="xkcd:orangey yellow",
        edgecolor='xkcd:dark grey',
        label="YSG",
        zorder=8,
    )

    # RSG comparison samples
    ax_hrd.scatter(
        np.log10(rsg_smc_df["Temp"]),
        rsg_smc_df["LogLum"],
        alpha=0.3,
        c="lightgrey",
        label="RSG",
        zorder=6,
        marker="v",
    )
    ax_hrd.scatter(
        np.log10(rsg_lmc_df["Temp"]),
        rsg_lmc_df["LogLum"],
        alpha=0.3,
        c="lightgrey",
        zorder=6,
        marker="v",
    )

    # sn_zorder = 9
    # sn_color = "xkcd:red"
    # sn_ecolor = "black"
    # # ax_hrd.errorbar(3.825, 5.0, xerr=0.025, yerr=0.05, fmt="o", color=sn_color, ecolor=sn_ecolor, label="SNIIb \nProgenitor", zorder=sn_zorder)
    # # ax_hrd.errorbar(3.72, 5.17, xerr=0.08, yerr=0.04, fmt="o", color=sn_color, ecolor=sn_ecolor, zorder=sn_zorder)
    # # ax_hrd.errorbar(3.98, 5.14, xerr=[[0.16], [0.21]], yerr=[[0.39], [0.22]], fmt="o", color=sn_color, ecolor=sn_ecolor, zorder=sn_zorder)
    # # ax_hrd.errorbar(3.63, 4.94, xerr=0.01, yerr=0.06, fmt="o", color=sn_color, ecolor=sn_ecolor, zorder=sn_zorder)
    # # ax_hrd.errorbar(3.78, 4.92, xerr=0.02, yerr=0.2, fmt="o", color=sn_color, ecolor=sn_ecolor, zorder=sn_zorder)
    # # ax_hrd.errorbar(3.63, 5.1, xerr=0.05, yerr=0.3, fmt="o", color=sn_color, ecolor=sn_ecolor, zorder=sn_zorder)
    # ax_hrd.scatter(3.825, 5.0, marker="X", s=200, color=sn_color, edgecolors=sn_ecolor, zorder=sn_zorder + 1, label="Known SN \nProgenitor")
    # ax_hrd.scatter(3.72, 5.17, marker="X", s=200, color=sn_color, edgecolors=sn_ecolor, zorder=sn_zorder + 1)
    # ax_hrd.scatter(3.98, 5.14, marker="X", s=200, color=sn_color, edgecolors=sn_ecolor, zorder=sn_zorder + 1)
    # ax_hrd.scatter(3.63, 4.94, marker="X", s=200, color=sn_color, edgecolors=sn_ecolor, zorder=sn_zorder + 1)
    # ax_hrd.scatter(3.78, 4.92, marker="X", s=200, color=sn_color, edgecolors=sn_ecolor, zorder=sn_zorder + 1)

    # Proposed targets
    for star_idx in proposed_indices:
        if star_idx not in var_df.index:
            continue
        if star_idx == 307:
            ax_hrd.scatter(
                var_df.loc[star_idx, "logT"],
                var_df.loc[star_idx, "logL"],
                marker="*",
                s=400,
                c="cyan",
                linewidths=2.0,
                edgecolors="black",
                label="Proposed \nTarget",
                zorder=100,
            )
        else:
            ax_hrd.scatter(
                var_df.loc[star_idx, "logT"],
                var_df.loc[star_idx, "logL"],
                marker="*",
                s=400,
                c="cyan",
                linewidths=2.0,
                edgecolors="black",
                zorder=100,
            )

    box_star_indices = [307, 780, 716]
    box_star_colours = ["xkcd:very light green", "xkcd:very light purple", "xkcd:light peach"]
    box_star_ecolors = ["green", "xkcd:violet", "xkcd:burnt orange"]
    box_dx = 0.06
    box_dy = 0.14
    for box_idx, facecol, ecol in zip(box_star_indices, box_star_colours, box_star_ecolors):
        if box_idx not in var_df.index:
            continue
        x0b = var_df.loc[box_idx, "logT"]
        y0b = var_df.loc[box_idx, "logL"]
        if pd.isna(x0b) or pd.isna(y0b):
            continue
        ax_hrd.add_patch(
            Rectangle(
                (x0b - box_dx / 2, y0b - box_dy / 2),
                box_dx,
                box_dy,
                facecolor=facecol,
                edgecolor=ecol,
                alpha=0.9,
                zorder=2,
                linewidth=1.5
            )
        )

    ax_hrd.invert_xaxis()
    ax_hrd.set_xlim(*hrd_xlim)
    ax_hrd.set_ylim(*hrd_ylim)
    ax_hrd.set_ylim(3.95,5.3)
    ax_hrd.tick_params(axis="both", direction="in", labelsize=22)
    ax_hrd.set_xlabel("log(T/[K])", fontsize=26)
    ax_hrd.set_ylabel("log(L/[L$_☉$])", fontsize=26)

    # Fixed-order legend
    handles, labels = ax_hrd.get_legend_handles_labels()
    label_to_handle = {}
    for h, lab in zip(handles, labels):
        if lab and (lab not in label_to_handle):
            label_to_handle[lab] = h

    wanted = ["YSG", "RSG", "Proposed \nCandidate"] #"Known SN \nProgenitor",
    ordered_handles = [label_to_handle[l] for l in wanted if l in label_to_handle]
    ordered_labels = [l for l in wanted if l in label_to_handle]

    leg = ax_hrd.legend(
        ordered_handles,
        ordered_labels,
        bbox_to_anchor=(1.125, 0.0),
        loc="lower right",
        title_fontsize=16,
        fontsize=22,
        markerscale=1.5,
        frameon=True,
        framealpha=1.0,
        facecolor="white",
        edgecolor="black",
    )
    leg.set_zorder(200)

    fig.savefig(savepath, dpi=dpi, bbox_inches="tight")
plot_hrd(#evol_files_smc[:-3][::-1],
    evol_files_smc[:7][::-1],
    savepath="hr_diagrams/plot_HRD_cropped_allYSGs.png", dpi=600,
    report=False)

In [ ]:
def plot_lc(
    *,
    open_file_func=None,
    var_df=None,
    rsg_smc_df=None,
    rsg_lmc_df=None,
    boxes=None,
    box_styles=None,
    proposed_indices=None,
    individual_plotter_func=None,
    figsize=(10, 10),
    width_ratios=(1.15, 1.0),
    hrd_xlim=(4.12, 3.40),
    hrd_ylim=(3.93, 5.48),
    ysg_box=(3.62, 4.00, 4.00, 5.48),
    savepath=None,
    dpi=300,
    report=False,):

    if open_file_func is None:
        open_file_func = Open_File
    if var_df is None:
        var_df = var
    if rsg_smc_df is None:
        rsg_smc_df = rsg_smc
    if rsg_lmc_df is None:
        rsg_lmc_df = rsg_lmc
    if boxes is None:
        boxes = [box3, box1, box2]
    if proposed_indices is None:
        proposed_indices = proposed
    if individual_plotter_func is None:
        individual_plotter_func = individual_plotter

    if box_styles is None:
        box_styles = [
            {"border_color": "xkcd:burnt orange", "bg": "xkcd:light peach", "border_width": 3.0},
            {"border_color": "green", "bg": "xkcd:very light green", "border_width": 3.0},
            {"border_color": "xkcd:violet", "bg": "xkcd:very light purple", "border_width": 3.0}
        ]

    if len(boxes) != 3:
        raise ValueError("Expected exactly 3 boxes (e.g., [box1, box2, box3])")

    fig = plt.figure(figsize=(12,10))
    fig, ax_lc = plt.subplots(nrows=3, ncols=1, figsize=(12, 10), sharex=True)

    # gs_right = gs[0, 1].subgridspec(3, 1, hspace=0.08)
    # ax_lc = [fig.add_subplot(gs_right[i]) for i in range(3)]

    for i, (indices, style) in enumerate(zip(boxes, box_styles)):
        individual_plotter_func(
            indices,
            g=True,
            seeoutliers=False,
            # report=report,
            bg=style.get("bg"),
            # bg_alpha=0.25,
            border_color=style.get("border_color"),
            border_width=style.get("border_width", 3.0),
            ax=ax_lc[i],
            show=False,
            tick_labelsize=22,
            label_fontsize=26,
            marker_size=3,
            # star_color_other='xkcd:yellow tan'
            
        )

        ax_lc[i].yaxis.set_major_locator(MultipleLocator(0.2))
        ax_lc[i].yaxis.set_minor_locator(AutoMinorLocator(2)) 
        ax_lc[i].tick_params(axis="y", which="minor", direction="in", length=3)     
        # set x lim
        ax_lc[i].set_xlim(-100, 3000)
        

        if i < 2:
            ax_lc[i].set_xlabel("")
            ax_lc[i].tick_params(labelbottom=False)
        else:
            ax_lc[i].yaxis.set_major_locator(MultipleLocator(0.4))
            ax_lc[i].yaxis.set_minor_locator(AutoMinorLocator(4)) 
            ax_lc[i].tick_params(axis="y", which="minor", direction="in", length=3) 
    fig.tight_layout()

    if savepath is not None:
        fig.savefig(savepath, dpi=dpi, bbox_inches="tight")

    return fig, ax_lc

plot_lc(savepath="hr_diagrams/plot_LC.png", dpi = 600)

In [ ]:
def plot_hrd_left_and_lc_boxes_right(
    evol_track_files,
    *,
    open_file_func=None,
    var_df=None,
    rsg_smc_df=None,
    rsg_lmc_df=None,
    boxes=None,
    box_styles=None,
    proposed_indices=None,
    individual_plotter_func=None,
    figsize=(22, 10),
    width_ratios=(1.15, 1.0),
    hrd_xlim=(4.12, 3.40),
    hrd_ylim=(3.93, 5.48),
    ysg_box=(3.62, 4.00, 4.00, 5.48),
    savepath=None,
    dpi=300,
    report=False,
):

    if open_file_func is None:
        open_file_func = Open_File
    if var_df is None:
        var_df = var
    if rsg_smc_df is None:
        rsg_smc_df = rsg_smc
    if rsg_lmc_df is None:
        rsg_lmc_df = rsg_lmc
    if boxes is None:
        boxes = [box3, box1, box2]
    if proposed_indices is None:
        proposed_indices = proposed
    if individual_plotter_func is None:
        individual_plotter_func = individual_plotter

    if box_styles is None:
        box_styles = [
            {"border_color": "xkcd:burnt orange", "bg": "xkcd:light peach", "border_width": 3.0},
            {"border_color": "green", "bg": "xkcd:very light green", "border_width": 3.0},
            {"border_color": "xkcd:violet", "bg": "xkcd:very light purple", "border_width": 3.0}
        ]

    if len(boxes) != 3:
        raise ValueError("Expected exactly 3 boxes (e.g., [box1, box2, box3])")

    fig = plt.figure(figsize=figsize)
    # Slightly larger gap between the left (HRD) and right (LCs) columns.
    gs = GridSpec(1, 2, figure=fig, width_ratios=width_ratios, wspace=0.31)

    # -----------------------------
    # Left panel: HRD
    # -----------------------------
    ax_hrd = fig.add_subplot(gs[0, 0])

    cmap = cm.inferno
    norm = mcolors.Normalize(vmin=0, vmax=len(evol_track_files) - 1)
    new_cmap = truncate_colormap(cmap, 0.0, 0.8, n=len(evol_track_files))

    # Tracks + mass labels
    for i, file_i in enumerate(evol_track_files):
        df_i = open_file_func(file_i)
        phase_i = get_phase(df_i)
        color = new_cmap(norm(i))

        ax_hrd.plot(
            phase_i["log_Teff"],
            phase_i["log_L"],
            "-",
            color=color,
            alpha=0.7,
            zorder=3,
        )

        x_label = 4.069
        idx_near = (phase_i["log_Teff"] - x_label).abs().idxmin()
        y_label = phase_i.loc[idx_near, "log_L"]

        # Mass annotation 
        ax_hrd.annotate(
            str(int(file_i.split("/")[-1].split(".track")[0][1:3])) + r" M$_\odot$",
            xy=(x_label, y_label),
            xytext=(0, 2),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=20,
            color=color,
            zorder=20,
            alpha=0.8,
            annotation_clip=False,
        )

    x0, x1, y0, y1 = ysg_box
    mask = var_df[
        (var_df["logT"] > x0)
        & (var_df["logT"] < x1)
        & (var_df["logL"] > y0)
        & (var_df["logL"] < y1)
        & (var_df["alarm_level_flag"] == 0)
    ]
    # ax_hrd.fill_betweenx([y0, y1], x0, x1, color="xkcd:yellow tan", alpha=0.2, zorder=1)
    ax_hrd.add_patch(Rectangle((3.62, 4.0),0.38,1.6,facecolor='none',edgecolor='black',
                alpha=0.5,
                zorder=20,
                linewidth=1.2,
                linestyle='dotted',
            ))

    ax_hrd.scatter(
        mask["logT"],
        mask["logL"],
        alpha=0.5,
        # color="xkcd:butter yellow",
        color='grey',
        # edgecolors="xkcd:orangey yellow",
        edgecolor='xkcd:dark grey',
        label="YSG",
        zorder=8,
    )

    # RSG comparison samples
    ax_hrd.scatter(
        np.log10(rsg_smc_df["Temp"]),
        rsg_smc_df["LogLum"],
        alpha=0.3,
        c="lightgrey",
        label="RSG",
        zorder=6,
        marker="v",
    )
    ax_hrd.scatter(
        np.log10(rsg_lmc_df["Temp"]),
        rsg_lmc_df["LogLum"],
        alpha=0.3,
        c="lightgrey",
        zorder=6,
        marker="v",
    )

    sn_zorder = 9
    sn_color = "xkcd:red"
    sn_ecolor = "black"
    # ax_hrd.errorbar(3.825, 5.0, xerr=0.025, yerr=0.05, fmt="o", color=sn_color, ecolor=sn_ecolor, label="SNIIb \nProgenitor", zorder=sn_zorder)
    # ax_hrd.errorbar(3.72, 5.17, xerr=0.08, yerr=0.04, fmt="o", color=sn_color, ecolor=sn_ecolor, zorder=sn_zorder)
    # ax_hrd.errorbar(3.98, 5.14, xerr=[[0.16], [0.21]], yerr=[[0.39], [0.22]], fmt="o", color=sn_color, ecolor=sn_ecolor, zorder=sn_zorder)
    # ax_hrd.errorbar(3.63, 4.94, xerr=0.01, yerr=0.06, fmt="o", color=sn_color, ecolor=sn_ecolor, zorder=sn_zorder)
    # ax_hrd.errorbar(3.78, 4.92, xerr=0.02, yerr=0.2, fmt="o", color=sn_color, ecolor=sn_ecolor, zorder=sn_zorder)
    # ax_hrd.errorbar(3.63, 5.1, xerr=0.05, yerr=0.3, fmt="o", color=sn_color, ecolor=sn_ecolor, zorder=sn_zorder)
    ax_hrd.scatter(3.825, 5.0, marker="X", s=200, color=sn_color, edgecolors=sn_ecolor, zorder=sn_zorder + 1, label="Known SN \nProgenitor")
    ax_hrd.scatter(3.72, 5.17, marker="X", s=200, color=sn_color, edgecolors=sn_ecolor, zorder=sn_zorder + 1)
    ax_hrd.scatter(3.98, 5.14, marker="X", s=200, color=sn_color, edgecolors=sn_ecolor, zorder=sn_zorder + 1)
    ax_hrd.scatter(3.63, 4.94, marker="X", s=200, color=sn_color, edgecolors=sn_ecolor, zorder=sn_zorder + 1)
    ax_hrd.scatter(3.78, 4.92, marker="X", s=200, color=sn_color, edgecolors=sn_ecolor, zorder=sn_zorder + 1)

    # Proposed targets
    for star_idx in proposed_indices:
        if star_idx not in var_df.index:
            continue
        if star_idx == 307:
            ax_hrd.scatter(
                var_df.loc[star_idx, "logT"],
                var_df.loc[star_idx, "logL"],
                marker="*",
                s=400,
                c="cyan",
                linewidths=2.0,
                edgecolors="black",
                label="Proposed \nTarget",
                zorder=100,
            )
        else:
            ax_hrd.scatter(
                var_df.loc[star_idx, "logT"],
                var_df.loc[star_idx, "logL"],
                marker="*",
                s=400,
                c="cyan",
                linewidths=2.0,
                edgecolors="black",
                zorder=100,
            )

    box_star_indices = [307, 780, 716]
    box_star_colours = ["xkcd:very light green", "xkcd:very light purple", "xkcd:light peach"]
    box_star_ecolors = ["green", "xkcd:violet", "xkcd:burnt orange"]
    box_dx = 0.06
    box_dy = 0.14
    for box_idx, facecol, ecol in zip(box_star_indices, box_star_colours, box_star_ecolors):
        if box_idx not in var_df.index:
            continue
        x0b = var_df.loc[box_idx, "logT"]
        y0b = var_df.loc[box_idx, "logL"]
        if pd.isna(x0b) or pd.isna(y0b):
            continue
        ax_hrd.add_patch(
            Rectangle(
                (x0b - box_dx / 2, y0b - box_dy / 2),
                box_dx,
                box_dy,
                facecolor=facecol,
                edgecolor=ecol,
                alpha=0.9,
                zorder=2,
                linewidth=1.5
            )
        )

    ax_hrd.invert_xaxis()
    ax_hrd.set_xlim(*hrd_xlim)
    ax_hrd.set_ylim(*hrd_ylim)
    ax_hrd.tick_params(axis="both", direction="in", labelsize=22)
    ax_hrd.set_xlabel("log(T/[K])", fontsize=26)
    ax_hrd.set_ylabel("log(L/[L$_☉$])", fontsize=26)

    # Fixed-order legend
    handles, labels = ax_hrd.get_legend_handles_labels()
    label_to_handle = {}
    for h, lab in zip(handles, labels):
        if lab and (lab not in label_to_handle):
            label_to_handle[lab] = h

    wanted = ["YSG", "RSG", "Known SN \nProgenitor", "Proposed \nTarget"]
    ordered_handles = [label_to_handle[l] for l in wanted if l in label_to_handle]
    ordered_labels = [l for l in wanted if l in label_to_handle]

    leg = ax_hrd.legend(
        ordered_handles,
        ordered_labels,
        bbox_to_anchor=(1.125, 0.0),
        loc="lower right",
        title_fontsize=16,
        fontsize=20,
        markerscale=1.5,
        frameon=True,
        framealpha=1.0,
        facecolor="white",
        edgecolor="black",
    )
    leg.set_zorder(200)

    # -----------------------------
    # Right panel: 3 stacked light-curve axes
    # -----------------------------
    gs_right = gs[0, 1].subgridspec(3, 1, hspace=0.08)
    ax_lc = [fig.add_subplot(gs_right[i, 0]) for i in range(3)]

    for i, (indices, style) in enumerate(zip(boxes, box_styles)):
        individual_plotter_func(
            indices,
            g=True,
            seeoutliers=False,
            report=report,
            bg=style.get("bg"),
            # bg_alpha=0.25,
            border_color=style.get("border_color"),
            border_width=style.get("border_width", 3.0),
            ax=ax_lc[i],
            show=False,
            tick_labelsize=22,
            label_fontsize=26,
            marker_size=3,
            # star_color_other='xkcd:yellow tan'
            
        )

        ax_lc[i].yaxis.set_major_locator(MultipleLocator(0.2))
        ax_lc[i].yaxis.set_minor_locator(AutoMinorLocator(2)) 
        ax_lc[i].tick_params(axis="y", which="minor", direction="in", length=3)     
        

        if i < 2:
            ax_lc[i].set_xlabel("")
            ax_lc[i].tick_params(labelbottom=False)
        else:
            ax_lc[i].yaxis.set_major_locator(MultipleLocator(0.4))
            ax_lc[i].yaxis.set_minor_locator(AutoMinorLocator(4)) 
            ax_lc[i].tick_params(axis="y", which="minor", direction="in", length=3) 
    fig.tight_layout()

    if savepath is not None:
        fig.savefig(savepath, dpi=dpi, bbox_inches="tight")

    return fig, ax_hrd, ax_lc


fig, ax_hrd, ax_lc = plot_hrd_left_and_lc_boxes_right(
    evol_files_smc[:-3][::-1],
    savepath="hr_diagrams/HRD_plus_lightcurves_side_by_side.png",
    report=False,
)


In [ ]:
def plot_hrd(evol_track_files,
    *,
    open_file_func=None,
    var_df=None,
    rsg_smc_df=None,
    rsg_lmc_df=None,
    boxes=None,
    box_styles=None,
    proposed_indices=None,
    individual_plotter_func=None,
    figsize=(10, 10),
    width_ratios=(1.15, 1.0),
    hrd_xlim=(4.12, 3.40),
    hrd_ylim=(3.93, 5.48),
    ysg_box=(3.62, 4.00, 4.00, 5.48),
    savepath=None,
    dpi=300,
    report=False,
):
    if open_file_func is None:
        open_file_func = Open_File
    if var_df is None:
        var_df = var
    if rsg_smc_df is None:
        rsg_smc_df = rsg_smc
    if rsg_lmc_df is None:
        rsg_lmc_df = rsg_lmc
    if boxes is None:
        boxes = [box3, box1, box2]
    if proposed_indices is None:
        proposed_indices = proposed
    if individual_plotter_func is None:
        individual_plotter_func = individual_plotter

    if box_styles is None:
        box_styles = [
            {"border_color": "xkcd:burnt orange", "bg": "xkcd:light peach", "border_width": 3.0},
            {"border_color": "green", "bg": "xkcd:very light green", "border_width": 3.0},
            {"border_color": "xkcd:violet", "bg": "xkcd:very light purple", "border_width": 3.0}
        ]

    if len(boxes) != 3:
        raise ValueError("Expected exactly 3 boxes (e.g., [box1, box2, box3])")
    # fig = plt.figure(figsize=figsize)
    fig, ax_hrd = plt.subplots(figsize=figsize)
    # Slightly larger gap between the left and right columns.
    gs = GridSpec(1, 2, figure=fig, width_ratios=width_ratios, wspace=0.31)

    # -----------------------------
    # Left panel: HRD
    # -----------------------------
    # ax_hrd = fig.add_subplot(gs[0, 0])

    cmap = cm.inferno
    norm = mcolors.Normalize(vmin=0, vmax=len(evol_track_files) - 1)
    new_cmap = truncate_colormap(cmap, 0.0, 0.8, n=len(evol_track_files))

    for i, file_i in enumerate(evol_track_files):
        df_i = open_file_func(file_i)
        phase_i = get_phase(df_i)
        color = new_cmap(norm(i))

        ax_hrd.plot(
            phase_i["log_Teff"],
            phase_i["log_L"],
            "-",
            color=color,
            alpha=0.7,
            zorder=3,
        )

        x_label = 4.069
        idx_near = (phase_i["log_Teff"] - x_label).abs().idxmin()
        y_label = phase_i.loc[idx_near, "log_L"]

        # Mass annotation
        ax_hrd.annotate(
            str(int(file_i.split("/")[-1].split(".track")[0][1:3])) + r" M$_\odot$",
            xy=(x_label, y_label),
            xytext=(0, 2),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=20,
            color=color,
            zorder=20,
            alpha=0.8,
            annotation_clip=False,
        )


    x0, x1, y0, y1 = ysg_box
    mask = var_df[
        (var_df["logT"] > x0)
        & (var_df["logT"] < x1)
        & (var_df["logL"] > y0)
        & (var_df["logL"] < y1)
        # & (var_df["alarm_level_flag"] == 0)
        # & (var_df["alarm_level_flag"].notna())
    ]
    # ax_hrd.fill_betweenx([y0, y1], x0, x1, color="xkcd:yellow tan", alpha=0.2, zorder=1)
    ax_hrd.add_patch(Rectangle((3.62, 4.0),0.38,1.6,facecolor='none',edgecolor='black',
                alpha=0.5,
                zorder=20,
                linewidth=1.2,
                linestyle='dotted',
            ))

    ax_hrd.scatter(
        mask["logT"],
        mask["logL"],
        alpha=0.5,
        # color="xkcd:butter yellow",
        color='grey',
        # edgecolors="xkcd:orangey yellow",
        edgecolor='xkcd:grey',
        label="YSG",
        zorder=8,
    )

    # RSG comparison samples
    ax_hrd.scatter(
        np.log10(rsg_smc_df["Temp"]),
        rsg_smc_df["LogLum"],
        alpha=0.3,
        c="lightgrey",
        label="RSG",
        zorder=6,
        marker="v",
    )
    ax_hrd.scatter(
        np.log10(rsg_lmc_df["Temp"]),
        rsg_lmc_df["LogLum"],
        alpha=0.3,
        c="lightgrey",
        zorder=6,
        marker="v",
    )

    # sn_zorder = 9
    # sn_color = "xkcd:red"
    # sn_ecolor = "black"
    # # ax_hrd.errorbar(3.825, 5.0, xerr=0.025, yerr=0.05, fmt="o", color=sn_color, ecolor=sn_ecolor, label="SNIIb \nProgenitor", zorder=sn_zorder)
    # # ax_hrd.errorbar(3.72, 5.17, xerr=0.08, yerr=0.04, fmt="o", color=sn_color, ecolor=sn_ecolor, zorder=sn_zorder)
    # # ax_hrd.errorbar(3.98, 5.14, xerr=[[0.16], [0.21]], yerr=[[0.39], [0.22]], fmt="o", color=sn_color, ecolor=sn_ecolor, zorder=sn_zorder)
    # # ax_hrd.errorbar(3.63, 4.94, xerr=0.01, yerr=0.06, fmt="o", color=sn_color, ecolor=sn_ecolor, zorder=sn_zorder)
    # # ax_hrd.errorbar(3.78, 4.92, xerr=0.02, yerr=0.2, fmt="o", color=sn_color, ecolor=sn_ecolor, zorder=sn_zorder)
    # # ax_hrd.errorbar(3.63, 5.1, xerr=0.05, yerr=0.3, fmt="o", color=sn_color, ecolor=sn_ecolor, zorder=sn_zorder)
    # ax_hrd.scatter(3.825, 5.0, marker="X", s=200, color=sn_color, edgecolors=sn_ecolor, zorder=sn_zorder + 1, label="Known SN \nProgenitor")
    # ax_hrd.scatter(3.72, 5.17, marker="X", s=200, color=sn_color, edgecolors=sn_ecolor, zorder=sn_zorder + 1)
    # ax_hrd.scatter(3.98, 5.14, marker="X", s=200, color=sn_color, edgecolors=sn_ecolor, zorder=sn_zorder + 1)
    # ax_hrd.scatter(3.63, 4.94, marker="X", s=200, color=sn_color, edgecolors=sn_ecolor, zorder=sn_zorder + 1)
    # ax_hrd.scatter(3.78, 4.92, marker="X", s=200, color=sn_color, edgecolors=sn_ecolor, zorder=sn_zorder + 1)

    # Proposed targets
    for star_idx in proposed_indices:
        if star_idx not in var_df.index:
            continue
        if star_idx == 307:
            ax_hrd.scatter(
                var_df.loc[star_idx, "logT"],
                var_df.loc[star_idx, "logL"],
                marker="*",
                s=400,
                c="cyan",
                linewidths=2.0,
                edgecolors="black",
                label="Proposed \nCandidate",
                zorder=100,
            )
        else:
            ax_hrd.scatter(
                var_df.loc[star_idx, "logT"],
                var_df.loc[star_idx, "logL"],
                marker="*",
                s=400,
                c="cyan",
                linewidths=2.0,
                edgecolors="black",
                zorder=100,
            )

    box_star_indices = [307, 780, 716]
    box_star_colours = ["xkcd:very light green", "xkcd:very light purple", "xkcd:light peach"]
    box_star_ecolors = ["green", "xkcd:violet", "xkcd:burnt orange"]
    box_dx = 0.06
    box_dy = 0.14
    for box_idx, facecol, ecol in zip(box_star_indices, box_star_colours, box_star_ecolors):
        if box_idx not in var_df.index:
            continue
        x0b = var_df.loc[box_idx, "logT"]
        y0b = var_df.loc[box_idx, "logL"]
        if pd.isna(x0b) or pd.isna(y0b):
            continue
        ax_hrd.add_patch(
            Rectangle(
                (x0b - box_dx / 2, y0b - box_dy / 2),
                box_dx,
                box_dy,
                facecolor=facecol,
                edgecolor=ecol,
                alpha=0.9,
                zorder=2,
                linewidth=1.5
            )
        )

    ax_hrd.invert_xaxis()
    ax_hrd.set_xlim(*hrd_xlim)
    ax_hrd.set_ylim(*hrd_ylim)
    ax_hrd.set_ylim(3.95,5.3)
    ax_hrd.tick_params(axis="both", direction="in", labelsize=22)
    ax_hrd.set_xlabel("log(T/[K])", fontsize=26)
    ax_hrd.set_ylabel("log(L/[L$_☉$])", fontsize=26)

    # Fixed-order legend
    handles, labels = ax_hrd.get_legend_handles_labels()
    label_to_handle = {}
    for h, lab in zip(handles, labels):
        if lab and (lab not in label_to_handle):
            label_to_handle[lab] = h

    wanted = ["YSG", "RSG", "Proposed \nCandidate"] #"Known SN \nProgenitor",
    ordered_handles = [label_to_handle[l] for l in wanted if l in label_to_handle]
    ordered_labels = [l for l in wanted if l in label_to_handle]

    leg = ax_hrd.legend(
        ordered_handles,
        ordered_labels,
        bbox_to_anchor=(1.125, 0.0),
        loc="lower right",
        title_fontsize=16,
        fontsize=22,
        markerscale=1.5,
        frameon=True,
        framealpha=1.0,
        facecolor="white",
        edgecolor="grey",
        # fontcolor="white"
    )
    leg.set_zorder(200)
    # plt.style.use('dark_background')
    fig.savefig(savepath, dpi=dpi, bbox_inches="tight")
plot_hrd(#evol_files_smc[:-3][::-1],
    evol_files_smc[:7][::-1],
    savepath="hr_diagrams/plot_HRD_cropped_allYSGs.png", dpi=600,
    report=False)

In [ ]:
def plot_hrd(evol_track_files,
    *,
    open_file_func=None,
    var_df=None,
    rsg_smc_df=None,
    rsg_lmc_df=None,
    boxes=None,
    box_styles=None,
    proposed_indices=None,
    individual_plotter_func=None,
    figsize=(10, 10),
    width_ratios=(1.15, 1.0),
    hrd_xlim=(4.12, 3.40),
    hrd_ylim=(3.93, 5.48),
    ysg_box=(3.62, 4.00, 4.00, 5.48),
    savepath=None,
    dpi=300,
    report=False,
):
    if open_file_func is None:
        open_file_func = Open_File
    if var_df is None:
        var_df = var
    if rsg_smc_df is None:
        rsg_smc_df = rsg_smc
    if rsg_lmc_df is None:
        rsg_lmc_df = rsg_lmc
    if boxes is None:
        boxes = [box3, box1, box2]
    if proposed_indices is None:
        proposed_indices = proposed
    if individual_plotter_func is None:
        individual_plotter_func = individual_plotter

    if box_styles is None:
        box_styles = [
            {"border_color": "xkcd:burnt orange", "bg": "xkcd:peach", "border_width": 3.0},
            {"border_color": "green", "bg": "xkcd:green", "border_width": 3.0},
            {"border_color": "xkcd:violet", "bg": "xkcd:purple", "border_width": 3.0}
        ]

    if len(boxes) != 3:
        raise ValueError("Expected exactly 3 boxes (e.g., [box1, box2, box3])")
    # fig = plt.figure(figsize=figsize)
    fig, ax_hrd = plt.subplots(figsize=figsize)
    # Slightly larger gap between the left and right columns.
    gs = GridSpec(1, 2, figure=fig, width_ratios=width_ratios, wspace=0.31)

    # -----------------------------
    # Left panel: HRD
    # -----------------------------
    # ax_hrd = fig.add_subplot(gs[0, 0])

    cmap = cm.Purples
    norm = mcolors.Normalize(vmin=0, vmax=len(evol_track_files) - 1)
    new_cmap = truncate_colormap(cmap, 0., 0.6, n=len(evol_track_files))

    for i, file_i in enumerate(evol_track_files):
        df_i = open_file_func(file_i)
        phase_i = get_phase(df_i)
        color = new_cmap(norm(i))

        ax_hrd.plot(
            phase_i["log_Teff"],
            phase_i["log_L"],
            "-",
            color=color,
            alpha=0.7,
            zorder=1,
        )

        x_label = 4.069
        idx_near = (phase_i["log_Teff"] - x_label).abs().idxmin()
        y_label = phase_i.loc[idx_near, "log_L"]

        # Mass annotation
        ax_hrd.annotate(
            str(int(file_i.split("/")[-1].split(".track")[0][1:3])) + r" M$_\odot$",
            xy=(x_label, y_label),
            xytext=(0, 2),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=20,
            color=color,
            zorder=20,
            alpha=0.8,
            annotation_clip=False,
        )


    x0, x1, y0, y1 = ysg_box
    mask = var_df[
        (var_df["logT"] > x0)
        & (var_df["logT"] < x1)
        & (var_df["logL"] > y0)
        & (var_df["logL"] < y1)
        # & (var_df["alarm_level_flag"] == 0)
        # & (var_df["alarm_level_flag"].notna())
    ]
    # ax_hrd.fill_betweenx([y0, y1], x0, x1, color="xkcd:yellow tan", alpha=0.2, zorder=1)
    ax_hrd.add_patch(Rectangle((3.62, 4.0),0.38,1.6,facecolor='none',edgecolor='black',
                alpha=0.5,
                zorder=20,
                linewidth=1.2,
                linestyle='dotted',
            ))

    ax_hrd.scatter(
        mask["logT"],
        mask["logL"],
        alpha=0.5,
        # color="xkcd:butter yellow",
        color='lightgrey',
        # edgecolors="xkcd:orangey yellow",
        edgecolor='white',
        label="YSG",
        zorder=8,
    )

    # RSG comparison samples
    ax_hrd.scatter(
        np.log10(rsg_smc_df["Temp"]),
        rsg_smc_df["LogLum"],
        alpha=0.4,
        c="grey",
        label="RSG",
        zorder=6,
        marker="v",
    )
    ax_hrd.scatter(
        np.log10(rsg_lmc_df["Temp"]),
        rsg_lmc_df["LogLum"],
        alpha=0.4,
        c="grey",
        zorder=6,
        marker="v",
    )

    # sn_zorder = 9
    # sn_color = "xkcd:red"
    # sn_ecolor = "black"
    # # ax_hrd.errorbar(3.825, 5.0, xerr=0.025, yerr=0.05, fmt="o", color=sn_color, ecolor=sn_ecolor, label="SNIIb \nProgenitor", zorder=sn_zorder)
    # # ax_hrd.errorbar(3.72, 5.17, xerr=0.08, yerr=0.04, fmt="o", color=sn_color, ecolor=sn_ecolor, zorder=sn_zorder)
    # # ax_hrd.errorbar(3.98, 5.14, xerr=[[0.16], [0.21]], yerr=[[0.39], [0.22]], fmt="o", color=sn_color, ecolor=sn_ecolor, zorder=sn_zorder)
    # # ax_hrd.errorbar(3.63, 4.94, xerr=0.01, yerr=0.06, fmt="o", color=sn_color, ecolor=sn_ecolor, zorder=sn_zorder)
    # # ax_hrd.errorbar(3.78, 4.92, xerr=0.02, yerr=0.2, fmt="o", color=sn_color, ecolor=sn_ecolor, zorder=sn_zorder)
    # # ax_hrd.errorbar(3.63, 5.1, xerr=0.05, yerr=0.3, fmt="o", color=sn_color, ecolor=sn_ecolor, zorder=sn_zorder)
    # ax_hrd.scatter(3.825, 5.0, marker="X", s=200, color=sn_color, edgecolors=sn_ecolor, zorder=sn_zorder + 1, label="Known SN \nProgenitor")
    # ax_hrd.scatter(3.72, 5.17, marker="X", s=200, color=sn_color, edgecolors=sn_ecolor, zorder=sn_zorder + 1)
    # ax_hrd.scatter(3.98, 5.14, marker="X", s=200, color=sn_color, edgecolors=sn_ecolor, zorder=sn_zorder + 1)
    # ax_hrd.scatter(3.63, 4.94, marker="X", s=200, color=sn_color, edgecolors=sn_ecolor, zorder=sn_zorder + 1)
    # ax_hrd.scatter(3.78, 4.92, marker="X", s=200, color=sn_color, edgecolors=sn_ecolor, zorder=sn_zorder + 1)

    # Proposed targets
    for star_idx in proposed_indices:
        if star_idx not in var_df.index:
            continue
        if star_idx == 307:
            ax_hrd.scatter(
                var_df.loc[star_idx, "logT"],
                var_df.loc[star_idx, "logL"],
                marker="*",
                s=650,
                c="xkcd:marigold",
                linewidths=2.0,
                edgecolors="xkcd:marigold",
                label="Proposed \nCandidate",
                zorder=100,
            )
        else:
            ax_hrd.scatter(
                var_df.loc[star_idx, "logT"],
                var_df.loc[star_idx, "logL"],
                marker="*",
                s=650,
                c="xkcd:marigold",
                linewidths=2.0,
                edgecolors="xkcd:marigold",
                zorder=100,
            )

    box_star_indices = [307, 780, 716]
    box_star_colours = ["xkcd:light teal", "xkcd:light lilac", "xkcd:dark sky blue"]
    box_star_ecolors = ["xkcd:teal", "xkcd:violet", "xkcd:blue"]
    box_dx = 0.06
    box_dy = 0.14
    for box_idx, facecol, ecol in zip(box_star_indices, box_star_colours, box_star_ecolors):
        if box_idx not in var_df.index:
            continue
        x0b = var_df.loc[box_idx, "logT"]
        y0b = var_df.loc[box_idx, "logL"]
        if pd.isna(x0b) or pd.isna(y0b):
            continue
        ax_hrd.add_patch(
            Rectangle(
                (x0b - box_dx / 2, y0b - box_dy / 2),
                box_dx,
                box_dy,
                facecolor=facecol,
                edgecolor=ecol,
                alpha=0.5,
                zorder=2,
                linewidth=1.5
            )
        )

    ax_hrd.invert_xaxis()
    ax_hrd.set_xlim(*hrd_xlim)
    ax_hrd.set_ylim(*hrd_ylim)
    ax_hrd.set_ylim(3.95,5.3)
    ax_hrd.tick_params(axis="both", direction="in", labelsize=22)
    ax_hrd.set_xlabel("log(T/[K])", fontsize=26)
    ax_hrd.set_ylabel("log(L/[L$_☉$])", fontsize=26)

    # Fixed-order legend
    handles, labels = ax_hrd.get_legend_handles_labels()
    label_to_handle = {}
    for h, lab in zip(handles, labels):
        if lab and (lab not in label_to_handle):
            label_to_handle[lab] = h

    wanted = ["YSG", "RSG", "Proposed \nCandidate"] #"Known SN \nProgenitor",
    ordered_handles = [label_to_handle[l] for l in wanted if l in label_to_handle]
    ordered_labels = [l for l in wanted if l in label_to_handle]

    leg = ax_hrd.legend(
        ordered_handles,
        ordered_labels,
        bbox_to_anchor=(1.125, 0.0),
        loc="lower right",
        title_fontsize=16,
        fontsize=22,
        markerscale=1.5,
        frameon=True,
        framealpha=1.0,
        facecolor="black",
        edgecolor="grey",
        # fontcolor="white"
    )
    leg.set_zorder(200)
    # plt.style.use('dark_background')
    fig.savefig(savepath, dpi=dpi, bbox_inches="tight")
plot_hrd(#evol_files_smc[:-3][::-1],
    evol_files_lmc[:6][::-1],
    savepath="hr_diagrams/plot_HRD_cropped_allYSGs.png", dpi=600,
    report=False)

In [ ]:
def plot_lc(
    *,
    open_file_func=None,
    var_df=None,
    rsg_smc_df=None,
    rsg_lmc_df=None,
    boxes=None,
    box_styles=None,
    proposed_indices=None,
    individual_plotter_func=None,
    figsize=(10, 10),
    width_ratios=(1.15, 1.0),
    hrd_xlim=(4.12, 3.40),
    hrd_ylim=(3.93, 5.48),
    ysg_box=(3.62, 4.00, 4.00, 5.48),
    savepath=None,
    dpi=300,
    report=False,):

    if open_file_func is None:
        open_file_func = Open_File
    if var_df is None:
        var_df = var
    if rsg_smc_df is None:
        rsg_smc_df = rsg_smc
    if rsg_lmc_df is None:
        rsg_lmc_df = rsg_lmc
    if boxes is None:
        boxes = [box3, box1, box2]
    if proposed_indices is None:
        proposed_indices = proposed
    if individual_plotter_func is None:
        individual_plotter_func = individual_plotter

    if box_styles is None:
        box_styles = [
            {"border_color": "xkcd:blue", "bg": "xkcd:dark sky blue", "border_width": 3.0},
            {"border_color": "xkcd:teal", "bg": "xkcd:light teal", "border_width": 3.0},
            {"border_color": "xkcd:violet", "bg": "xkcd:light lilac", "border_width": 3.0},
            
        ]
        
    if len(boxes) != 3:
        raise ValueError("Expected exactly 3 boxes (e.g., [box1, box2, box3])")

    fig = plt.figure(figsize=(12,10))
    fig, ax_lc = plt.subplots(nrows=3, ncols=1, figsize=(12, 10), sharex=True)

    # gs_right = gs[0, 1].subgridspec(3, 1, hspace=0.08)
    # ax_lc = [fig.add_subplot(gs_right[i]) for i in range(3)]

    for i, (indices, style) in enumerate(zip(boxes, box_styles)):
        individual_plotter_func(
            indices,
            g=True,
            seeoutliers=False,
            # report=report,
            bg=style.get("bg"),
            bg_alpha=0.5,
            border_color=style.get("border_color"),
            border_width=style.get("border_width", 3.0),
            ax=ax_lc[i],
            show=False,
            tick_labelsize=22,
            label_fontsize=26,
            marker_size=3,
            # star_color_other='xkcd:yellow tan'
            
        )

        ax_lc[i].yaxis.set_major_locator(MultipleLocator(0.2))
        ax_lc[i].yaxis.set_minor_locator(AutoMinorLocator(2)) 
        ax_lc[i].tick_params(axis="y", which="minor", direction="in", length=3)     
        # set x lim
        ax_lc[i].set_xlim(-100, 3000)
        

        if i < 2:
            ax_lc[i].set_xlabel("")
            ax_lc[i].tick_params(labelbottom=False)
        else:
            ax_lc[i].yaxis.set_major_locator(MultipleLocator(0.4))
            ax_lc[i].yaxis.set_minor_locator(AutoMinorLocator(4)) 
            ax_lc[i].tick_params(axis="y", which="minor", direction="in", length=3) 
    fig.tight_layout()

    if savepath is not None:
        fig.savefig(savepath, dpi=dpi, bbox_inches="tight")

    return fig, ax_lc

plot_lc(savepath="hr_diagrams/plot_LC_marigold.png", dpi = 600)

In [ ]:
box1 = [307,286,756, 1189] #1168
# box4 = [1130,1182,911,978, 1101]
box2 = [716,541,579,1100]
box3 = [780,670,765, 378]
proposed = [307,1130,613,1085,716,953,250,890,156,40,780,522]


def individual_plotter(indices, tail=1, g=True, seeoutliers=False, report=True, bg=None, border_color=None, border_width=2.0):
    if isinstance(indices, (int, np.integer)):
        indices = [int(indices)]
    else:
        indices = [int(i) for i in indices]

    # Use Matplotlib's default color cycle (one color per star).
    cycle_colors = plt.rcParams['axes.prop_cycle'].by_key().get('color', [])
    if not cycle_colors:
        cycle_colors = ['C0', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9']

    band_label = 'g' if g else 'V'

    # First pass: load data and choose a global reference HJD0 so the x-axis is readable.
    loaded = []
    hjd0 = None
    for index in indices:
        # Filtered data (outliers removed by default).
        df, telescopes = offset_corrector(index, g=g, show=False)
        fulldata = None
        if seeoutliers:
            fulldata = df_extract(index, g=g, seeoutliers=True, report=False)[0]
        loaded.append((index, df, fulldata, telescopes))
        candidate = float(np.nanmin(df['HJD'].values))
        if fulldata is not None:
            candidate = min(candidate, float(np.nanmin(fulldata['HJD'].values)))
        hjd0 = candidate if (hjd0 is None) else min(hjd0, candidate)

    # plt.figure(figsize=(12, 20))
    plt.figure(figsize=(14, 5))
    if bg is not None:
        # plt.gcf().patch.set_facecolor(bg)
        plt.gca().set_facecolor(bg)
    if border_color is not None:
        fig = plt.gcf()
        ax = plt.gca()
        for spine in ax.spines.values():
            spine.set_color(border_color)
            spine.set_linewidth(border_width)
        # fig.patch.set_edgecolor(border_color)
        # fig.patch.set_linewidth(border_width)
        fig.set_frameon(True)
    for j, (index, df, fulldata, telescopes) in enumerate(loaded):
        RA = coords['RA'].iloc[index]
        dec = coords['DEC'].iloc[index]

        if report:
            print(f"\nIndex {index} ({RA} {dec})")
            print("Number of observations per telescope:")
            print(df.groupby('telescope').size())

        color = cycle_colors[j % len(cycle_colors)]
        t = df['HJD'] - hjd0
        if index in proposed:
            color = 'cyan'
        else:
            color = 'xkcd:sky'
            # color=color
        plt.errorbar(
            t,
            df['mag'],
            yerr=df['mag_err'],
            fmt='o',
            markeredgecolor='none',
            markersize=3,
            capsize=0,
            color=color,
            zorder=10,
            # label=f"{band_label}-band (idx {index})",
        )


    plt.xlabel(f'Days', fontsize=24)
    plt.ylabel('m$_g$ [mag]', fontsize=24)
    # plt.title(f"{band_label}-band light curves ({len(indices)} stars)")
    # plt.legend()
    # plt.grid(True)
    plt.gca().invert_yaxis()  # Invert y-axis since smaller magnitudes are brighter
    plt.tick_params(axis='both', direction='in', labelsize=20)
    # plt.savefig(f'figs/offsets_examples/{RA}{dec}_g.png')
    # plt.savefig('hr_diagrams/box3.png', bbox_inches='tight', dpi=300)
    plt.savefig('../figs/misc/CClc.png', bbox_inches='tight', dpi=300)
    # If saving, keep the chosen background (otherwise Matplotlib may default to white).
    # Example: plt.savefig('hr_diagrams/box3.png', bbox_inches='tight', dpi=300, facecolor=plt.gcf().get_facecolor())
    plt.show()

individual_plotter([19], g=True, seeoutliers=False, report=True)#, border_color='xkcd:burnt orange', bg = 'xkcd:light peach', border_width=3.0)
# individual_plotter([1025], g=True, seeoutliers=False, report=True)#, border_color='green', bg = 'xkcd:very light green', border_width=3.0)

In [ ]:
def plot_phase_fold(index, best_period, df=None, telescopes=None, g=True, phase_bins=2, correct_offsets=True, mag_space=False):
    """
    Phase fold time series data using a given period.
    
    Parameters:
    -----------
    time : array-like
        Time values
    period : float
        Period to fold by
    values : array-like or None
        Data values to fold
    errors : array-like or None
        Uncertainties in data values
    phase_bins : float
        Number of phase cycles to show (e.g., 2 shows 0-2 phases)
        
    Returns:
    --------
    dict : Dictionary containing phased data
    """
    if df is None and correct_offsets == False:
        df, telescopes = df_extract(index, g=g)
    if df is None and correct_offsets == True:
        df, telescopes = offset_corrector(index, additive=False, show=False)
    RA = coords['RA'].iloc[index]
    dec = coords['DEC'].iloc[index]
    time = np.array(df['HJD'])  # Time in HJD
    flux = np.array(df['flux_(mJy)'])  # Flux in mJy
    mags = np.array(df['mag'])  # Magnitudes
    flux_errors = np.array(df['flux_err'])  # Flux uncertainties in mJy
    mag_errors = np.array(df['mag_err'])  # Magnitude uncertainties
    # period = lombs['best_period']  # Best period from Lomb-Scargle analysis
    # period = 30.89
    if best_period > (time.max() - time.min()):
        return None
    # Calculate phase
    colors = ['g', 'b', 'r', 'c', 'm', 'y', 'k']
    phased_time = (time / best_period) % phase_bins

    fig, ax = plt.subplots(figsize=(5, 6))
    plt.style.use('dark_background')
    # colour by telescope:
    for i, telescope in enumerate(telescopes):
        mask = df['telescope'] == telescope
        if mag_space:
            ax.errorbar(phased_time[mask], mags[mask], yerr=mag_errors[mask],  fmt='o', label=f'Telescope {telescope}', color="xkcd:sky")
            # bin and plot median in each phase bin for magnitudes:
        else:
            ax.errorbar(phased_time[mask], flux[mask], yerr=flux_errors[mask], fmt='o', label=f'Telescope {telescope}', color="xkcd:sky")
    # if mag_space:
    #     num_bins = int(np.ceil(len(phased_time) / 10))
    #     bins = np.linspace(0, phase_bins, num_bins + 1)
    #     bin_indices = np.digitize(phased_time, bins) - 1
    #     bin_centers = (bins[:-1] + bins[1:]) / 2
    #     mean_mags = [np.mean(mags[bin_indices == j]) for j in range(num_bins)]
    #     plt.plot(bin_centers, mean_mags, color='black', linestyle='-', marker='s', markersize=6, label=f'Mean {telescope}', zorder = 12)
    # plt.legend(fontsize=12)
    # single colour:
    # plt.errorbar(phased_time, flux, yerr=errors, fmt='o', alpha=0.7, markersize=4)
    ax.set_xlabel('Phase', fontsize=24)
    if mag_space:
        ax.set_ylabel('m$_g$ [mag]', fontsize=24)
        ax.invert_yaxis()  # Invert y-axis for magnitudes
    else:
        ax.set_ylabel('Flux (mJy)', fontsize=24)
    ax.tick_params(axis='both', which='major', direction ='in', labelsize=18)
    # ax.grid(True, alpha=0.3)
    # ax.title(f'{RA} {dec}; {index} (2 phases with period = {best_period:.2f} days)', fontsize=14)
    plt.tight_layout()
    # plt.savefig(f'figs/lc_plots/{RA}{dec}_phaseg.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'../figs/misc/{RA}{dec}_phaseg.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Bin and find amplitude of narrow bin
    num_bins = int(np.ceil(len(phased_time) / 10))  # e.g., ~10 points per bin
    bins = np.linspace(0, phase_bins, num_bins + 1)
    bin_indices = np.digitize(phased_time, bins) - 1

    amplitudes = []
    for i in range(num_bins):
        bin_flux = flux[bin_indices == i]
        if len(bin_flux) > 1:
            amplitude = np.max(bin_flux) - np.min(bin_flux)
            amplitudes.append(amplitude)
    average_amplitude = np.mean(amplitudes)
    print(f"Average scatter (amplitude) per 10 point phase bin: {average_amplitude:.3f} mJy")
    return average_amplitude

plot_phase_fold(19, 28.447711922617962, g=True, correct_offsets=True, mag_space=True)
# plot_phase_fold(1025, 761.28, g=True, correct_offsets=True, mag_space=True)


In [ ]:
from metrics import compute_lomb_scargle
def plot_periodogram(index, frequency, power, best_freq, false_alarm_level, title="Lomb-Scargle Periodogram", auto=False, show_best=True):
    """
    Plot the Lomb-Scargle periodogram.
    
    Parameters:
    -----------
    frequency : array-like
        Frequency values
    power : array-like
        Power values
    title : str
        Plot title
    freq_range : tuple or None
        (min_freq, max_freq) for x-axis limits
    show_best : bool
        Whether to mark the highest power peak
    """
    RA = coords['RA'].iloc[index]
    dec = coords['DEC'].iloc[index]
    print(f"Grid sampled at {len(frequency)} points")
    fig = plt.figure(figsize=(5, 6))
    plt.plot(frequency, power, color='xkcd:sky', linestyle='-', linewidth=2)
    # plt.ylim(0, np.max(power) * 1.1)  # Set y-axis limit to 10% above max power
    
    if show_best:
        best_idx = np.where(frequency == best_freq)[0][0]  # Find index in frequency array
        plt.axvline(x=frequency[best_idx], color='xkcd:light red', linestyle='--', alpha=0.7, 
                   label=f'Best Period = \n{1/frequency[best_idx]:.1f} days')
        plt.plot(frequency[best_idx], power[best_idx], 'ro', markersize=8)
        # for peak in peaks:
        #     if peak != frequency[best_idx]:  # Exclude the tallest peak
        #         plt.axvline(x=peak, color='green', linestyle='--', alpha=0.5,
        #                    label=f'Peak at {1/peak:.2f} days')
    
    if auto is not True:
        # plt.xlim(0, 0.9)
        plt.xlim(0, 0.2)
    
    if show_best:
        plt.legend(fontsize=18)
    plt.axhline(y=false_alarm_level, color='grey', linestyle='--',
                label=f'False Alarm Level ({100*0.01:.1f}%): {false_alarm_level:.3e}')
    plt.xlabel('Frequency [1/day]', fontsize=24)
    plt.ylabel('Power', fontsize=24)
    plt.tick_params(axis='both', which='major', direction ='in', labelsize=18)
    # plt.grid(True, alpha=0.3)
    # plt.title(f'{RA} {dec} periodogram', fontsize=14)
    # plt.tight_layout()
    # plt.savefig(f'figs/lc_plots/{RA}{dec}_periodogramg.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'../figs/misc/{RA}{dec}_periodogramg.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)

index = 19
df, telescopes = offset_corrector(index, additive=False, show=False)
lombs = compute_lomb_scargle(index, df=df, telescopes=telescopes, g=True, auto=False, samples_per_peak=10)
plot_periodogram(index, lombs['frequency'], lombs['power'], lombs['best_frequency'], lombs['false_alarm_level'], lombs['peaks'], lombs['observation_period'], show_best=True)

In [ ]:
import math

import matplotlib.pyplot as plt

import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle


def plot_colortable(colors, *, ncols=4, sort_colors=True):

    cell_width = 212
    cell_height = 22
    swatch_width = 48
    margin = 12

    # Sort colors by hue, saturation, value and name.
    if sort_colors is True:
        names = sorted(
            colors, key=lambda c: tuple(mcolors.rgb_to_hsv(mcolors.to_rgb(c))))
    else:
        names = list(colors)

    n = len(names)
    nrows = math.ceil(n / ncols)

    width = cell_width * ncols + 2 * margin
    height = cell_height * nrows + 2 * margin
    dpi = 72

    fig, ax = plt.subplots(figsize=(width / dpi, height / dpi), dpi=dpi)
    fig.subplots_adjust(margin/width, margin/height,
                        (width-margin)/width, (height-margin)/height)
    ax.set_xlim(0, cell_width * ncols)
    ax.set_ylim(cell_height * (nrows-0.5), -cell_height/2.)
    ax.yaxis.set_visible(False)
    ax.xaxis.set_visible(False)
    ax.set_axis_off()

    for i, name in enumerate(names):
        row = i % nrows
        col = i // nrows
        y = row * cell_height

        swatch_start_x = cell_width * col
        text_pos_x = cell_width * col + swatch_width + 7

        ax.text(text_pos_x, y, name, fontsize=14,
                horizontalalignment='left',
                verticalalignment='center')

        ax.add_patch(
            Rectangle(xy=(swatch_start_x, y-9), width=swatch_width,
                      height=18, facecolor=colors[name], edgecolor='0.7')
        )

    return fig
xkcd_fig = plot_colortable(mcolors.XKCD_COLORS)
xkcd_fig.savefig("XKCD_Colors.png")